In [4]:
# -*- coding: utf-8 -*-
"""
Reddit Sampling Pipeline
--------------------------------------------------------
Builds a Reddit corpus similar to the AARP dataset sampling pattern.
This version ignores all timestamps to avoid UTC or date window issues.
"""

from __future__ import annotations

import hashlib
from typing import Iterable, List, Optional, Tuple, Dict
import numpy as np
import pandas as pd

# Optional NLP dependencies
_HAS_SBERT = False
_HAS_LANGDETECT = False

try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity
    _HAS_SBERT = True
except Exception:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

try:
    from langdetect import detect
    _HAS_LANGDETECT = True
except Exception:
    pass


# ------------------------
# Helpers
# ------------------------

TEXT_CANDIDATES = [
    "body", "text", "content", "message", "comment_body", "body_text", "body_markdown"
]

def _resolve_text_col(df: pd.DataFrame) -> str:
    """Return the first available text column name in df."""
    for c in TEXT_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(
        f"Couldn't find a text column in df_comments. "
        f"Looked for: {TEXT_CANDIDATES}. "
        f"Available columns: {list(df.columns)}"
    )

def _hash_text(s: str) -> str:
    """Stable hash for deduplication of similar comments."""
    norm = " ".join((s or "").split()).strip().lower()
    return hashlib.sha256(norm.encode("utf-8", errors="ignore")).hexdigest()

def _is_english(text: str) -> bool:
    """Detect English language if possible; otherwise, accept all."""
    if not text:
        return False
    if _HAS_LANGDETECT:
        try:
            return detect(text) == "en"
        except Exception:
            return True
    return True


# ------------------------
# Topics and Embeddings
# ------------------------

DEFAULT_TOPICS = [
    "The government should not forgive student loan debt.",
    "Airbnb should be banned in cities.",
    "The federal minimum wage should be increased.",
    "The US should provide financial and military aid to Ukraine.",
    "A universal basic income would kill the economy.",
    "Climate change is one of the greatest threats to humanity.",
    "Fur clothing should be banned.",
    "The government should not invest in renewable energy.",
    "There should only be vegetarian food in cantines.",
    "Gender-neutral language and stating pronouns are silly issues.",
    "Prostitution should be illegal.",
    "Employers should mandate vaccination.",
    "The government should not be responsible for universal health care.",
    "Immigrants should adopt the local language and culture.",
    "We need stricter gun control laws.",
    "The death penalty should be reestablished.",
    "Police officers should wear body cameras.",
    "Artificial Intelligence should replace humans where possible.",
    "Social media is a threat to democracy.",
]


def _build_topic_vector(topics: List[str]):
    """Build a mean topic embedding using SBERT or TF-IDF fallback."""
    if _HAS_SBERT:
        model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        topic_emb = model.encode(topics, show_progress_bar=False)
        topic_vec = np.mean(topic_emb, axis=0, keepdims=True)
        return ("sbert", model, topic_vec)
    else:
        vec = TfidfVectorizer(min_df=2, max_df=0.9, ngram_range=(1, 2))
        X = vec.fit_transform(topics)
        topic_centroid = np.asarray(X.mean(axis=0))
        return ("tfidf", vec, topic_centroid)


def _subreddit_vectors(
    df_comments: pd.DataFrame,
    df_submissions: pd.DataFrame,
    model_tuple,
    per_sub_sample_comments: int = 1000,
) -> Dict[str, np.ndarray]:
    """Compute a text vector per subreddit using comments and titles."""
    kind, model, _ = model_tuple
    sub_vectors = {}
    text_col = _resolve_text_col(df_comments)

    subs_basic = (
        df_submissions[["subreddit", "title", "selftext"]].copy()
        if "subreddit" in df_submissions
        else pd.DataFrame()
    )
    coms_basic = df_comments[["subreddit", text_col]].copy()

    for sub, g in coms_basic.groupby("subreddit"):
        g_sample = (
            g.sample(min(len(g), per_sub_sample_comments), random_state=42)
            if len(g) > 0
            else g
        )

        texts: List[str] = list(g_sample[text_col].astype(str).values)

        if not subs_basic.empty:
            ssub = subs_basic[subs_basic["subreddit"] == sub].head(200)
            if "title" in ssub:
                texts.extend(list(ssub["title"].astype(str).values))
            if "selftext" in ssub:
                texts.extend(list(ssub["selftext"].astype(str).values))

        if not texts:
            continue

        if kind == "sbert":
            emb = model.encode(texts, show_progress_bar=False)
            sub_vectors[sub] = np.mean(emb, axis=0, keepdims=True)
        else:
            vec = TfidfVectorizer(min_df=5, max_df=0.95, ngram_range=(1, 2))
            X = vec.fit_transform(texts)
            sub_vectors[sub] = np.asarray(X.mean(axis=0))

    return sub_vectors


def _rank_subreddits_by_similarity(model_tuple, topic_vec, sub_vectors: Dict[str, np.ndarray]) -> pd.DataFrame:
    """Compute cosine similarity subreddit vs. topic and rank."""
    sims = []
    for sub, vec in sub_vectors.items():
        try:
            sim = float(cosine_similarity(topic_vec, vec)[0][0])
        except Exception:
            sim = np.nan
        sims.append((sub, sim))

    rank = (
        pd.DataFrame(sims, columns=["subreddit", "cosine_similarity"])
        .sort_values("cosine_similarity", ascending=False)
        .reset_index(drop=True)
    )
    return rank


# ------------------------
# Filters and Thread Selection
# ------------------------

def _apply_activity_filter_alltime(df_submissions: pd.DataFrame, min_posts_total: int = 500) -> set:
    """Keep only subreddits with enough submissions across the entire dataset."""
    counts = df_submissions.groupby("subreddit").size()
    return set(counts[counts >= min_posts_total].index)


def _apply_size_filter(df_sub_meta: Optional[pd.DataFrame],
                       min_subscribers: int = 50_000,
                       max_subscribers: int = 2_000_000) -> set:
    """Filter by subreddit size if metadata available."""
    if df_sub_meta is None or not {"subreddit", "subscribers"} <= set(df_sub_meta.columns):
        return set()
    ok = df_sub_meta[
        (df_sub_meta["subscribers"] >= min_subscribers)
        & (df_sub_meta["subscribers"] <= max_subscribers)
    ]
    return set(ok["subreddit"].unique())


def _pick_threads_no_week(df_submissions: pd.DataFrame,
                          candidate_subs: Iterable[str],
                          top_k_per_subreddit: int = 50) -> pd.DataFrame:
    """Pick top-K submissions per subreddit by comment count."""
    dfx = df_submissions.copy()
    dfx = dfx[dfx["subreddit"].isin(set(candidate_subs))]

    if "num_comments" not in dfx.columns:
        dfx["num_comments"] = dfx.get("comments_count", 0)

    out = (
        dfx.sort_values("num_comments", ascending=False)
           .groupby(["subreddit"], as_index=False)
           .head(top_k_per_subreddit)
           .reset_index(drop=True)
    )
    return out


# ------------------------
# Main Sampler
# ------------------------

def sample_reddit_corpus(
    df_submissions: pd.DataFrame,
    df_comments: pd.DataFrame,
    topics: Optional[List[str]] = None,
    min_posts_activity: int = 500,
    per_sub_sample_comments: int = 100000,
    target_n_subs: Tuple[int, int] = (8, 12),
    top_k_threads_per_subreddit: int = 50,
    english_only: bool = True,
    deduplicate_comments: bool = True,
    df_subreddit_meta: Optional[pd.DataFrame] = None,
    min_subscribers: int = 50_000,
    max_subscribers: int = 2_000_000,
    max_comments_per_thread: Optional[int] = None,
    random_state: int = 42,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Build a Reddit sample
    """
    np.random.seed(random_state)

    if "subreddit" not in df_submissions.columns or "subreddit" not in df_comments.columns:
        raise ValueError("Both dataframes must have a 'subreddit' column.")

    # 1) Topic vector
    topics = topics or DEFAULT_TOPICS
    model_tuple = _build_topic_vector(topics)
    topic_vec = model_tuple[2]

    # 2) Subreddit vectors and ranking
    sub_vecs = _subreddit_vectors(
        df_comments=df_comments,
        df_submissions=df_submissions,
        model_tuple=model_tuple,
        per_sub_sample_comments=per_sub_sample_comments,
    )
    ranked = _rank_subreddits_by_similarity(model_tuple, topic_vec, sub_vecs)

    # 3) Apply filters (activity + size)
    size_ok = _apply_size_filter(df_subreddit_meta, min_subscribers, max_subscribers)
    if size_ok:
        ranked["size_ok"] = ranked["subreddit"].isin(size_ok)
    else:
        ranked["size_ok"] = True

    activity_ok = _apply_activity_filter_alltime(df_submissions, min_posts_total=min_posts_activity)
    ranked["activity_ok"] = ranked["subreddit"].isin(activity_ok)

    ranked["passes_filters"] = ranked["size_ok"] & ranked["activity_ok"]
    allowed = ranked[ranked["passes_filters"]].copy()

    low, high = target_n_subs
    chosen = allowed.head(high) if len(allowed) >= low else allowed.head(low)
    candidate_subs = list(chosen["subreddit"].values)

    # 4) Pick threads (top-K per subreddit overall)
    sampled_threads = _pick_threads_no_week(
        df_submissions=df_submissions,
        candidate_subs=candidate_subs,
        top_k_per_subreddit=top_k_threads_per_subreddit,
    )

    # Normalize submission_id
    st = sampled_threads.copy()
    if "submission_id" in st.columns:
        st["submission_id"] = st["submission_id"].astype(str)
    else:
        st["submission_id"] = st["id"].astype(str)

    # 5) Collect comments for selected threads (no time filter)
    dc = df_comments.copy()

    if "submission_id" not in dc.columns:
        if "link_id" in dc.columns:
            dc["submission_id"] = dc["link_id"].astype(str).str.replace(r"^t3_", "", regex=True)
        elif "link_id_fullname" in dc.columns:
            dc["submission_id"] = dc["link_id_fullname"].astype(str).str.replace(r"^t3_", "", regex=True)
        elif "parent_id" in dc.columns:
            dc["submission_id"] = dc["parent_id"].astype(str).str.replace(r"^t3_", "", regex=True)
        else:
            raise ValueError("df_comments needs a submission linkage (submission_id/link_id).")

    dc = dc[dc["submission_id"].isin(set(st["submission_id"]))]

    # 6) English-only + deduplication
    text_col = _resolve_text_col(dc)

    if english_only:
        dc["is_english"] = dc[text_col].astype(str).map(_is_english)
        dc = dc[dc["is_english"]]

    if max_comments_per_thread is not None:
        dc = (
            dc.sort_values("score" if "score" in dc.columns else text_col, ascending=False)
              .groupby("submission_id", as_index=False)
              .head(max_comments_per_thread)
        )

    if deduplicate_comments:
        dc["body_hash"] = dc[text_col].astype(str).map(_hash_text)
        dc = dc.drop_duplicates(subset=["submission_id", "body_hash"])

    # 7) Build final outputs (no date fields)
    sampled_comments = dc[[
        c for c in [
            "submission_id", "subreddit", "author",
            text_col, "score",
            "parent_id" if "parent_id" in dc.columns else None,
            "is_submitter" if "is_submitter" in dc.columns else None,
            "depth" if "depth" in dc.columns else None,
        ] if c is not None and c in dc.columns
    ]].copy().rename(columns={text_col: "body"})

    sampled_threads_tidy = st[[
        c for c in [
            "submission_id", "id" if "id" in st.columns else None,
            "subreddit", "title",
            "selftext" if "selftext" in st.columns else None,
            "num_comments", "score" if "score" in st.columns else None,
            "author" if "author" in st.columns else None,
        ] if c is not None and c in st.columns
    ]].copy()

    ranked_subs = ranked.copy()
    return ranked_subs, sampled_threads_tidy, sampled_comments


# ------------------------
# Example usage
# ------------------------

if __name__ == "__main__":
    from pathlib import Path

    p_sub = Path("data/submissions")
    p_com = Path("data/comments")

    print("Exists submissions:", p_sub.exists(), p_sub.resolve())
    print("Exists comments   :", p_com.exists(), p_com.resolve())

    if p_sub.exists():
        df_submissions = pd.read_json(p_sub, lines=True)
    else:
        df_submissions = pd.DataFrame()

    if p_com.exists():
        df_comments = pd.read_json(p_com, lines=True)
    else:
        df_comments = pd.DataFrame()

    # Run sampler (no date filters)
    ranked_subs, sampled_threads, sampled_comments = sample_reddit_corpus(
        df_submissions=df_submissions,
        df_comments=df_comments,
        english_only=True,
        deduplicate_comments=True,
        top_k_threads_per_subreddit=50,
        random_state=42,
    )

    # Save output
    out_dir = Path("out_sampling")
    out_dir.mkdir(parents=True, exist_ok=True)
    ranked_subs.to_json(out_dir / "ranked_subreddits.ndjson", lines=True, orient="records", force_ascii=False)
    sampled_threads.to_json(out_dir / "sampled_threads.ndjson", lines=True, orient="records", force_ascii=False)
    sampled_comments.to_json(out_dir / "sampled_comments.ndjson", lines=True, orient="records", force_ascii=False)

    print("Sampling complete. Output written to", out_dir)


Exists submissions: True /Users/arthur/DataspellProjects/reddit-l/data/submissions
Exists comments   : True /Users/arthur/DataspellProjects/reddit-l/data/comments
Sampling complete. Output written to out_sampling
